Roadmap item 6: sweep theta_consistency and theta_confidence, report
accuracy/precision/recall/F1 (overall AND per-category) at each setting,
using the already-computed S_E/S_C/S_D/C_f/Conf_f from verification_results.csv.

This does NOT re-run the graph verification — it re-applies only the final
hierarchical decision rule at different thresholds, which is legitimate since
C_f and Conf_f are threshold-independent scores; only the DECISION changes.

In [15]:
import pandas as pd
import numpy as np
from pathlib import Path

INTERMEDIATE_DIR = Path("../Datasets/intermediate")
RESULTS_PATH = INTERMEDIATE_DIR / "verification_results.csv"
SWEEP_OUTPUT_PATH = INTERMEDIATE_DIR / "threshold_sweep_results.csv"
SWEEP_CATEGORY_OUTPUT_PATH = INTERMEDIATE_DIR / "threshold_sweep_by_category.csv"

THETA_CONSISTENCY_RANGE = np.round(np.arange(0.50, 0.95, 0.05), 2)
THETA_CONFIDENCE_RANGE = np.round(np.arange(0.30, 0.75, 0.05), 2)

In [16]:
def recompute_decision(row, theta_consistency, theta_confidence):
    """Re-apply the exact three-level hierarchy from detect_hallucination(),
    using the already-computed C_f / patterns_detected / Conf_f."""
    if row['C_f'] < theta_consistency:
        return 'HALLUCINATION'
    if row['patterns_detected'] != 'None' and pd.notna(row['patterns_detected']):
        return 'HALLUCINATION'
    if row['Conf_f'] < theta_confidence:
        return 'HALLUCINATION'
    return 'VERIFIED'


In [17]:
def compute_metrics(df, decision_col='decision_swept'):
    TP = len(df[(df.ground_truth == 'HALLUCINATION') & (df[decision_col] == 'HALLUCINATION')])
    TN = len(df[(df.ground_truth == 'VERIFIED') & (df[decision_col] == 'VERIFIED')])
    FP = len(df[(df.ground_truth == 'VERIFIED') & (df[decision_col] == 'HALLUCINATION')])
    FN = len(df[(df.ground_truth == 'HALLUCINATION') & (df[decision_col] == 'VERIFIED')])

    total = len(df)
    accuracy = (TP + TN) / total if total else 0
    precision = TP / (TP + FP) if (TP + FP) else 0
    recall = TP / (TP + FN) if (TP + FN) else 0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) else 0

    return {'TP': TP, 'TN': TN, 'FP': FP, 'FN': FN,
            'accuracy': accuracy, 'precision': precision, 'recall': recall, 'f1': f1}


In [18]:
def main():
    df = pd.read_csv(RESULTS_PATH)
    print(f"Loaded {len(df)} verification results")
    print(f"Sweeping theta_consistency: {list(THETA_CONSISTENCY_RANGE)}")
    print(f"Sweeping theta_confidence: {list(THETA_CONFIDENCE_RANGE)}")

    sweep_rows = []
    category_sweep_rows = []

    for tc in THETA_CONSISTENCY_RANGE:
        for tf in THETA_CONFIDENCE_RANGE:
            df['decision_swept'] = df.apply(
                lambda r: recompute_decision(r, tc, tf), axis=1
            )
            m = compute_metrics(df)
            sweep_rows.append({
                'theta_consistency': tc, 'theta_confidence': tf, **m
            })

            # per-category accuracy at this threshold setting
            for cat in df['category'].unique():
                cat_df = df[df['category'] == cat]
                cat_correct = (cat_df['ground_truth'] == cat_df['decision_swept']).sum()
                cat_total = len(cat_df)
                category_sweep_rows.append({
                    'theta_consistency': tc, 'theta_confidence': tf,
                    'category': cat, 'n': cat_total,
                    'accuracy': cat_correct / cat_total if cat_total else 0
                })

    sweep_df = pd.DataFrame(sweep_rows)
    category_sweep_df = pd.DataFrame(category_sweep_rows)

    sweep_df.to_csv(SWEEP_OUTPUT_PATH, index=False)
    category_sweep_df.to_csv(SWEEP_CATEGORY_OUTPUT_PATH, index=False)

    print(f"\nSaved overall sweep: {SWEEP_OUTPUT_PATH} ({len(sweep_df)} rows)")
    print(f"Saved per-category sweep: {SWEEP_CATEGORY_OUTPUT_PATH} ({len(category_sweep_df)} rows)")

    # quick summary: best F1 overall, and what domain_violation accuracy looks like there
    best_row = sweep_df.loc[sweep_df['f1'].idxmax()]
    print(f"\n--- Best overall F1 ---")
    print(best_row)

    best_tc, best_tf = best_row['theta_consistency'], best_row['theta_confidence']
    best_cat_slice = category_sweep_df[
        (category_sweep_df.theta_consistency == best_tc) &
        (category_sweep_df.theta_confidence == best_tf)
        ]
    print(f"\n--- Per-category accuracy at best-F1 threshold (tc={best_tc}, tf={best_tf}) ---")
    print(best_cat_slice[['category', 'n', 'accuracy']].to_string(index=False))

    # also show the ORIGINAL fixed thresholds (0.8, 0.5) for direct comparison
    original_row = sweep_df[
        (sweep_df.theta_consistency == 0.80) & (sweep_df.theta_confidence == 0.50)
        ]
    if len(original_row) > 0:
        print(f"\n--- For comparison: original fixed thresholds (tc=0.80, tf=0.50) ---")
        print(original_row.to_string(index=False))
    else:
        print("\n(Note: tc=0.80 not in swept range if you changed THETA_CONSISTENCY_RANGE — "
              "add it explicitly if you want a direct before/after comparison.)")


In [19]:
main()

Loaded 974 verification results
Sweeping theta_consistency: [np.float64(0.5), np.float64(0.55), np.float64(0.6), np.float64(0.65), np.float64(0.7), np.float64(0.75), np.float64(0.8), np.float64(0.85), np.float64(0.9)]
Sweeping theta_confidence: [np.float64(0.3), np.float64(0.35), np.float64(0.4), np.float64(0.45), np.float64(0.5), np.float64(0.55), np.float64(0.6), np.float64(0.65), np.float64(0.7)]

Saved overall sweep: ../Datasets/intermediate/threshold_sweep_results.csv (81 rows)
Saved per-category sweep: ../Datasets/intermediate/threshold_sweep_by_category.csv (324 rows)

--- Best overall F1 ---
theta_consistency      0.500000
theta_confidence       0.550000
TP                   574.000000
TN                   315.000000
FP                    85.000000
FN                     0.000000
accuracy               0.912731
precision              0.871017
recall                 1.000000
f1                     0.931062
Name: 5, dtype: float64

--- Per-category accuracy at best-F1 threshold (

In [20]:
import pandas as pd

results_df = pd.read_csv("../Datasets/intermediate/verification_results.csv")

def recompute_decision(row, tc=0.5, tf=0.55):
    if row['C_f'] < tc:
        return 'HALLUCINATION'
    if row['patterns_detected'] != 'None' and pd.notna(row['patterns_detected']):
        return 'HALLUCINATION'
    if row['Conf_f'] < tf:
        return 'HALLUCINATION'
    return 'VERIFIED'

results_df['decision_best'] = results_df.apply(recompute_decision, axis=1)

dv = results_df[results_df.category == 'domain_violation']
wrong = dv[dv.decision_best != dv.ground_truth]

print(f"domain_violation misclassified: {len(wrong)}/{len(dv)}")
print(wrong[['subject', 'predicate', 'object', 'S_E', 'S_C', 'S_D', 'C_f', 'Conf_f']].to_string(index=False))

atc_overrides = pd.read_csv("../Datasets/intermediate/atc_entity_domain_overrides.csv")
atc_drugs = set(atc_overrides['entity_name'])
wrong_atc_involved = wrong[wrong.subject.isin(atc_drugs) | wrong.object.isin(atc_drugs)]
print(f"\nOf those, how many involve an ATC-grounded drug: {len(wrong_atc_involved)}")

domain_violation misclassified: 0/75
Empty DataFrame
Columns: [subject, predicate, object, S_E, S_C, S_D, C_f, Conf_f]
Index: []

Of those, how many involve an ATC-grounded drug: 0


In [21]:
import pandas as pd

INTERMEDIATE_DIR_STR = "../Datasets/intermediate"
test_facts = pd.read_csv(f"{INTERMEDIATE_DIR_STR}/test_facts_large_v2.csv")
entity_rel = pd.read_csv(f"{INTERMEDIATE_DIR_STR}/entity_relationships.csv")

real_edges = set(zip(entity_rel['subject'], entity_rel['predicate'], entity_rel['object']))

# also check reverse direction for symmetric interacts_with
real_edges_reverse = set(
    (o, p, s) for s, p, o in real_edges if p == 'interacts_with'
)
all_real = real_edges | real_edges_reverse

mislabeled = test_facts[
    (test_facts.ground_truth == 'HALLUCINATION') &
    test_facts.apply(lambda r: (r['subject'], r['predicate'], r['object']) in all_real, axis=1)
    ]

print(f"Total HALLUCINATION-labeled facts that are actually real edges: {len(mislabeled)}")
print(mislabeled.groupby('category').size())

Total HALLUCINATION-labeled facts that are actually real edges: 0
Series([], dtype: int64)
